In [1]:
import geopandas as gpd
import pandas as pd
import os

# Cargar polígonos reales de las provincias desde el shapefile de Natural Earth.
shapefile_path = '../data/external/ne_10m_admin_1_states_provinces.shp'
gdf_poligonos = gpd.read_file(shapefile_path)

# Filtrar solo Ecuador
gdf_ecuador = gdf_poligonos[gdf_poligonos['admin'] == 'Ecuador'].copy()
gdf_ecuador = gdf_ecuador.to_crs('EPSG:4326')

# Calcular centroides geométricos (punto representativo de cada provincia)
gdf_ecuador['centroid'] = gdf_ecuador.geometry.centroid
gdf_ecuador['centroid_lat'] = gdf_ecuador['centroid'].y
gdf_ecuador['centroid_lon'] = gdf_ecuador['centroid'].x

# Diccionario de capitales para mantener referencia histórica
capitales = {
    'Azuay': 'Cuenca', 'Bolívar': 'Guaranda', 'Cañar': 'Azogues',
    'Carchi': 'Tulcán', 'Chimborazo': 'Riobamba', 'Cotopaxi': 'Latacunga',
    'El Oro': 'Machala', 'Esmeraldas': 'Esmeraldas', 'Galápagos': 'Puerto Baquerizo Moreno',
    'Guayas': 'Guayaquil', 'Imbabura': 'Ibarra', 'Loja': 'Loja',
    'Los Ríos': 'Babahoyo', 'Manabí': 'Portoviejo', 'Morona Santiago': 'Macas',
    'Napo': 'Tena', 'Orellana': 'Puerto Francisco de Orellana',
    'Pastaza': 'Puyo', 'Pichincha': 'Quito', 'Santa Elena': 'Santa Elena',
    'Santo Domingo de los Tsáchilas': 'Santo Domingo',
    'Sucumbíos': 'Nueva Loja', 'Tungurahua': 'Ambato', 'Zamora Chinchipe': 'Zamora'
}

# Coordenadas de capitales (referencia histórica)
capital_coords = {
    'Azuay': (-2.9006, -79.0040), 'Bolívar': (-1.5928, -79.0440),
    'Cañar': (-2.7447, -78.8486), 'Carchi': (0.8119, -77.7176),
    'Chimborazo': (-1.6589, -78.9481), 'Cotopaxi': (-0.9333, -78.6153),
    'El Oro': (-3.9586, -79.5952), 'Esmeraldas': (0.9661, -79.6540),
    'Galápagos': (-0.9047, -89.6160), 'Guayas': (-2.1890, -79.8840),
    'Imbabura': (0.3517, -78.1225), 'Loja': (-3.9931, -79.2043),
    'Los Ríos': (-1.8022, -79.5344), 'Manabí': (-1.0546, -80.4544),
    'Morona Santiago': (-2.3066, -78.1161), 'Napo': (-0.9914, -77.8123),
    'Orellana': (-0.2926, -76.9876), 'Pastaza': (-1.4924, -78.0022),
    'Pichincha': (-0.2295, -78.5243), 'Santa Elena': (-2.2278, -80.8585),
    'Santo Domingo de los Tsáchilas': (-0.2530, -79.1753),
    'Sucumbíos': (0.0780, -76.8969), 'Tungurahua': (-1.2393, -78.6233),
    'Zamora Chinchipe': (-4.0689, -79.0015)
}

# Construir dataset con coordenadas de centroides (features del modelo) y capitales (referencia)
registros = []
for _, row in gdf_ecuador.iterrows():
    prov = row.get('name', '')
    if prov in capitales:
        cap_lat, cap_lon = capital_coords[prov]
    else:
        cap_lat, cap_lon = row['centroid_lat'], row['centroid_lon']
    registros.append({
        'provincia': prov,
        'capital': capitales.get(prov, ''),
        'centroid_lat': round(row['centroid_lat'], 4),
        'centroid_lon': round(row['centroid_lon'], 4),
        'capital_lat': cap_lat,
        'capital_lon': cap_lon
    })

df_provincias = pd.DataFrame(registros)
print('[OK] Dataset de provincias creado con centroides geométricos.')
print('\n--- Lista de Provincias ---')
print(df_provincias[['provincia', 'centroid_lat', 'centroid_lon']])

# Exportar GeoJSON con coordenadas de centroide (features) + capitales (referencia)
gdf_provinces = gpd.GeoDataFrame(
    df_provincias,
    geometry=gpd.points_from_xy(
        df_provincias['centroid_lon'], df_provincias['centroid_lat']
    ),
    crs='EPSG:4326'
)

os.makedirs('../data/external', exist_ok=True)
gdf_provinces.to_file('../data/external/ecuador_provincias_capitales.geojson', driver='GeoJSON')
print('[OK] GeoDataFrame guardado correctamente en ../data/external/ecuador_provincias_capitales.geojson')
print(f'CRS del GeoDataFrame: {gdf_provinces.crs}')
print(f'\nVerificación: {len(df_provincias)} provincias procesadas, {df_provincias["centroid_lat"].nunique()} coordenadas únicas.')

[OK] Dataset de provincias creado con centroides geométricos.

--- Lista de Provincias ---
                         provincia  centroid_lat  centroid_lon
0                       Esmeraldas        0.6955      -79.2240
1                           Carchi        0.7437      -78.0412
2                        Sucumbios       -0.0054      -76.5867
3                         Orellana       -0.7776      -76.3872
4                          Pastaza       -1.7072      -76.8878
5                  Morona Santiago       -2.5557      -78.0172
6                 Zamora Chinchipe       -4.1654      -78.9132
7                             Loja       -4.0952      -79.6647
8                           El Oro       -3.5060      -79.8454
9                           Guayas       -2.0798      -79.8972
10                       Galápagos       -0.5504      -90.9043
11                     Santa Elena       -2.1292      -80.5638
12                          Manabi       -0.7347      -80.1323
13                         

/var/folders/mh/p902nyz14477qbgspbxj31lw0000gn/T/ipykernel_7918/805841278.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_ecuador['centroid'] = gdf_ecuador.geometry.centroid
